# TTT LoRA Functional Test

Trains the model for ~2000 steps on a single GPU, then compares:
- `sliding_bpb` — standard sliding-window eval (baseline)
- `ttt_bpb` — test-time LoRA adaptation eval

**Expected outcome:** `ttt_bpb < sliding_bpb`

Works on: Kaggle (T4/P100), Google Colab (T4/A100)

## Cell 1 — GPU check

In [ ]:
import os, torch

assert torch.cuda.is_available(), "No CUDA GPU found — switch to a GPU runtime"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

IS_COLAB  = 'google.colab' in str(get_ipython())
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
platform  = 'Colab' if IS_COLAB else ('Kaggle' if IS_KAGGLE else 'Other')

print(f"Platform : {platform}")
print(f"GPU      : {gpu_name}")
print(f"VRAM     : {vram_gb:.1f} GB")

## Cell 2 — Clone repo

In [ ]:
import os

if not os.path.exists('parameter-golf'):
    !git clone https://github.com/openai/parameter-golf.git

os.chdir('parameter-golf')
print('Working directory:', os.getcwd())

## Cell 3 — Install dependencies

In [ ]:
!pip install -q sentencepiece huggingface-hub tqdm zstandard

In [ ]:
import torch

# Flash Attention 2 with GQA (enable_gqa=True) requires compute capability >= 8.0.
# T4 and P100 are sm75/sm60 — patch the script to enable mem_efficient/math as fallbacks.
sm_major = torch.cuda.get_device_properties(0).major
if sm_major < 8:
    print(f'GPU compute capability {sm_major}.x < 8.0 — patching SDP backends')
    script = 'records/track_10min_16mb/2026-03-22_TTT_LoRA/train_gpt.py'
    src = open(script).read()
    src = src.replace('enable_mem_efficient_sdp(False)', 'enable_mem_efficient_sdp(True)')
    src = src.replace('enable_math_sdp(False)', 'enable_math_sdp(True)')
    open(script, 'w').write(src)
    print('Patched: mem_efficient_sdp=True, math_sdp=True (flash still preferred when available)')
else:
    print(f'GPU compute capability {sm_major}.x >= 8.0 — no patch needed')

## Cell 4 — Download data (2 train shards + full val split, ~2 GB)

In [ ]:
# Each train shard is ~100M tokens (~1 GB). 2 shards gives enough variety for a test run.
# The full val split is always downloaded regardless of --train-shards.
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 2

import glob
train_shards = sorted(glob.glob('data/datasets/fineweb10B_sp1024/fineweb_train_*.bin'))
val_shards   = sorted(glob.glob('data/datasets/fineweb10B_sp1024/fineweb_val_*.bin'))
print(f'Train shards : {len(train_shards)}')
print(f'Val shards   : {len(val_shards)}')
assert train_shards and val_shards, 'Data download failed — check HuggingFace access'

import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

# Scale batch tokens to GPU memory.
# With WORLD_SIZE=1, grad_accum_steps=8, so each forward pass sees
# TRAIN_BATCH_TOKENS/8 tokens. Stay well under 16 GB VRAM on T4.
if vram_gb >= 35:        # A100
    batch_tokens = 524288
elif vram_gb >= 14:      # T4 / V100
    batch_tokens = 262144
else:                    # smaller (P100 12 GB)
    batch_tokens = 131072

print(f'VRAM={vram_gb:.0f} GB → TRAIN_BATCH_TOKENS={batch_tokens}')

env = {
    'DATA_PATH'            : './data/datasets/fineweb10B_sp1024',
    'TOKENIZER_PATH'       : './data/tokenizers/fineweb_1024_bpe.model',
    'VOCAB_SIZE'           : '1024',
    'ITERATIONS'           : '2000',
    'TRAIN_BATCH_TOKENS'   : str(batch_tokens),
    'TRAIN_SEQ_LEN'        : '1024',
    'MAX_WALLCLOCK_SECONDS': '3600',   # 1-hour hard cap
    'SEED'                 : '42',
    'TTT_ENABLED'          : '1',
    'VAL_LOSS_EVERY'       : '500',    # periodic val checkpoints
}

env_str = ' '.join(f'{k}={v}' for k, v in env.items())
script  = 'records/track_10min_16mb/2026-03-22_TTT_LoRA/train_gpt.py'

!{env_str} torchrun --standalone --nproc_per_node=1 {script} 2>&1 | tee train_log.txt

In [ ]:
import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

# Scale batch tokens to GPU memory.
# With WORLD_SIZE=1, grad_accum_steps=8, so each forward pass sees
# TRAIN_BATCH_TOKENS/8 tokens. Stay well under 16 GB VRAM on T4.
if vram_gb >= 35:        # A100
    batch_tokens = 524288
elif vram_gb >= 14:      # T4 / V100
    batch_tokens = 262144
else:                    # smaller (P100 12 GB)
    batch_tokens = 131072

print(f'VRAM={vram_gb:.0f} GB → TRAIN_BATCH_TOKENS={batch_tokens}')

env = {
    'DATA_PATH'            : './data/datasets/fineweb10B_sp1024',
    'TOKENIZER_PATH'       : './data/tokenizers/fineweb_1024_bpe.model',
    'VOCAB_SIZE'           : '1024',
    'ITERATIONS'           : '2000',
    'TRAIN_BATCH_TOKENS'   : str(batch_tokens),
    'TRAIN_SEQ_LEN'        : '1024',
    'MAX_WALLCLOCK_SECONDS': '3600',   # 1-hour hard cap
    'SEED'                 : '42',
    'TTT_ENABLED'          : '1',
    'VAL_LOSS_EVERY'       : '500',    # periodic val checkpoints
}

env_str = ' '.join(f'{k}={v}' for k, v in env.items())
script  = 'records/track_10min_16mb/2026-03-22_TTT_LoRA/train_gpt.py'

!{env_str} torchrun --standalone --nproc_per_node=1 {script} 2>&1 | tee train_log.txt

## Cell 6 — Parse and display results

In [ ]:
import re

log = open('train_log.txt').read()

# Find the final eval lines written at end of training
m_sliding = re.search(r'final_int8_zlib_roundtrip\s+val_loss:([\d.]+)\s+val_bpb:([\d.]+)', log)
m_ttt     = re.search(r'final_ttt_lora\s+val_loss:([\d.]+)\s+val_bpb:([\d.]+)', log)

if m_sliding and m_ttt:
    s_loss, s_bpb = float(m_sliding.group(1)), float(m_sliding.group(2))
    t_loss, t_bpb = float(m_ttt.group(1)),     float(m_ttt.group(2))
    delta = s_bpb - t_bpb
    status = 'PASS ✓  TTT improves BPB' if t_bpb < s_bpb else 'FAIL ✗  TTT did not improve'

    print('=' * 50)
    print(f'Sliding BPB  : {s_bpb:.4f}  (loss={s_loss:.4f})')
    print(f'TTT LoRA BPB : {t_bpb:.4f}  (loss={t_loss:.4f})')
    print(f'Delta        : {delta:+.4f}')
    print(f'Result       : {status}')
    print('=' * 50)
else:
    print('Could not parse final results from train_log.txt')
    print('Check that training completed. Last 20 lines:')
    print('\n'.join(log.strip().splitlines()[-20:]))